# 01 · GC10-DET Steel Defects - EDA

**Arkon Manufacturing | Department: Sheet Rolling**

Dataset: 10 classes of steel defects, bounding box annotations (XML).

| Folder | Defect class |
|---|---|
| 1 | Punching hole |
| 2 | Welding line |
| 3 | Crescent gap |
| 4 | Water spot |
| 5 | Oil spot |
| 6 | Silk spot |
| 7 | Inclusion |
| 8 | Rolled pit |
| 9 | Crease |
| 10 | Waist folding |

In [ ]:
import sys
from pathlib import Path

_nb_root = Path('../..').resolve()
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))

from utils.arkon_utils import (
    get_device, get_mlflow_uri, save_figure,
    Timer, CheckpointManager, recommended_num_workers
)
print('arkon_utils loaded ✓')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageDraw
import xml.etree.ElementTree as ET
import pandas as pd
from collections import Counter

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
ASSETS = 'cv/gc10'


In [ ]:
DATA_DIR  = Path('../../../data/06_gc10/raw')
LABEL_DIR = DATA_DIR / 'lable'  # XML annotations (typo in the original dataset)

# Mapping: folder → class name
CLASS_MAP = {
    '1': 'punching_hole', '2': 'welding_line', '3': 'crescent_gap',
    '4': 'water_spot',    '5': 'oil_spot',      '6': 'silk_spot',
    '7': 'inclusion',     '8': 'rolled_pit',     '9': 'crease',
    '10': 'waist_folding'
}

assert DATA_DIR.exists(), f'Not found: {DATA_DIR}'
print(f'Data dir: {DATA_DIR}')
print(f'Image folders: {sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir() and d.name.isdigit()])}')
print(f'Label files: {len(list(LABEL_DIR.glob("*.xml")))} XML files')

## 1. Image Count per Class

In [ ]:
rows = []
for folder_id, class_name in CLASS_MAP.items():
    img_dir = DATA_DIR / folder_id
    imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.bmp')) + list(img_dir.glob('*.png'))
    rows.append({'folder': folder_id, 'class': class_name, 'count': len(imgs)})

df = pd.DataFrame(rows).sort_values('folder')
print(df.to_string(index=False))
print(f"\nTotal images: {df['count'].sum()}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
colors = plt.cm.tab10(np.linspace(0, 1, len(df)))
ax.bar(df['class'], df['count'], color=colors)
ax.set_xticklabels(df['class'], rotation=35, ha='right')
ax.set_ylabel('Images')
for i, (_, row) in enumerate(df.iterrows()):
    ax.text(i, row['count'] + 2, str(row['count']), ha='center', fontsize=8)
plt.title('GC10-DET - Class Distribution (class imbalance!)')
plt.tight_layout()
save_figure(fig, 'cv_gc10_plot_1', subfolder='cv/gc10')
plt.show()
print(f"Imbalance ratio: {df['count'].max() / df['count'].min():.1f}x")

## 2. Sample Images per Defect Class

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for idx, (folder_id, class_name) in enumerate(CLASS_MAP.items()):
    row, col = divmod(idx, 5)
    img_dir = DATA_DIR / folder_id
    imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.bmp'))
    if imgs:
        img = Image.open(imgs[0]).convert('RGB')
        axes[row][col].imshow(img)
    axes[row][col].set_title(f'{folder_id}: {class_name}', fontsize=8)
    axes[row][col].axis('off')
plt.suptitle('GC10-DET - Sample Images per Defect Class')
plt.tight_layout()
save_figure(fig, 'cv_gc10_plot_2', subfolder='cv/gc10')
plt.show()

## 3. Annotation Analysis - Bounding Boxes

In [ ]:
def parse_gc10_xml(xml_path: Path) -> dict:
    tree = ET.parse(xml_path)
    root = tree.getroot()
    result = {'filename': root.findtext('filename', ''),
              'boxes': []}
    size = root.find('size')
    if size is not None:
        result['width']  = int(size.findtext('width',  0))
        result['height'] = int(size.findtext('height', 0))
    for obj in root.findall('object'):
        name = obj.findtext('name', '')
        bbox = obj.find('bndbox')
        if bbox is not None:
            result['boxes'].append({
                'class': name,
                'xmin': float(bbox.findtext('xmin', 0)),
                'ymin': float(bbox.findtext('ymin', 0)),
                'xmax': float(bbox.findtext('xmax', 0)),
                'ymax': float(bbox.findtext('ymax', 0)),
            })
    return result

# Parse first 100 XMLs
all_boxes = []
for xml_f in sorted(LABEL_DIR.glob('*.xml'))[:100]:
    parsed = parse_gc10_xml(xml_f)
    for box in parsed['boxes']:
        box['img_w'] = parsed.get('width', 0)
        box['img_h'] = parsed.get('height', 0)
        all_boxes.append(box)

df_boxes = pd.DataFrame(all_boxes)
print(f'Parsed {len(df_boxes)} boxes from 100 XMLs')
if not df_boxes.empty:
    df_boxes['box_w'] = df_boxes['xmax'] - df_boxes['xmin']
    df_boxes['box_h'] = df_boxes['ymax'] - df_boxes['ymin']
    print(df_boxes.groupby('class')[['box_w','box_h']].mean().round(1))

In [ ]:
# Visualize sample with bounding box
for folder_id in sorted(CLASS_MAP.keys())[:1]:
    img_dir = DATA_DIR / folder_id
    imgs = list(img_dir.glob('*.jpg'))
    if imgs:
        img = Image.open(imgs[0]).convert('RGB')
        # Try to find matching XML
        xml_path = LABEL_DIR / (imgs[0].stem + '.xml')
        if xml_path.exists():
            parsed = parse_gc10_xml(xml_path)
            draw = ImageDraw.Draw(img)
            for box in parsed['boxes']:
                draw.rectangle([box['xmin'], box['ymin'], box['xmax'], box['ymax']],
                               outline='red', width=3)
        plt.figure(figsize=(6, 5))
        plt.imshow(img)
        plt.title(f"GC10 Class {folder_id}: {CLASS_MAP[folder_id]}")
        plt.axis('off'); save_figure(fig, 'cv_gc10_plot_3', subfolder='cv/gc10')
plt.show()

## Summary

| Parameter | Value |
|---|---|
| Task | 10-class Classification |
| Note | **Strong imbalance** (class 6: 651 vs class 8: 31) |
| Annotations | Pascal VOC XML (detection) |
| Imbalance handling | WeightedRandomSampler or class_weight in the loss |

➡️ **Next step:** `02_gc10_preprocessing.ipynb`